# **Prévision explicative**

### Import des différents outils



In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.model_selection import GridSearchCV
from dotenv import load_dotenv
import os
import pandas as pd
load_dotenv()


True

### Import du dataset

In [ ]:
benin_data_path=os.getenv('BENIN_DATA_PATH')

benin_filtered= pd.read_csv(benin_data_path, sep=';')

benin_filtered

,Unnamed: 0,country_id,country_name,market_id,market_name,commodity_id,commodity_name,currency_id,currency_name,point_type_id,point_type_name,unit_id,unit_name,month,year,price,date
0,185502,29.0,Benin,1044,Malanville (CBM),51,Maize - Wholesale,0.0,XOF,14,Wholesale,5,KG,1,2002,145.0,2002-01-01
1,185503,29.0,Benin,1044,Malanville (CBM),51,Maize - Wholesale,0.0,XOF,14,Wholesale,5,KG,1,2003,106.0,2003-01-01
2,185504,29.0,Benin,1044,Malanville (CBM),51,Maize - Wholesale,0.0,XOF,14,Wholesale,5,KG,2,2003,107.5,2003-02-01
3,185505,29.0,Benin,1044,Malanville (CBM),51,Maize - Wholesale,0.0,XOF,14,Wholesale,5,KG,3,2003,95.0,2003-03-01
4,185506,29.0,Benin,1044,Malanville (CBM),51,Maize - Wholesale,0.0,XOF,14,Wholesale,5,KG,4,2003,95.0,2003-04-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39525,225027,29.0,Benin,2782,Zogbodomey,561,"Fish (fresh, silvi) - Retail",0.0,XOF,15,Retail,5,KG,3,2021,1200.0,2021-03-01
39526,225028,29.0,Benin,2782,Zogbodomey,561,"Fish (fresh, silvi) - Retail",0.0,XOF,15,Retail,5,KG,4,2021,1200.0,2021-04-01
39527,225029,29.0,Benin,2782,Zogbodomey,561,"Fish (fresh, silvi) - Retail",0.0,XOF,15,Retail,5,KG,5,2021,1200.0,2021-05-01
39528,225030,29.0,Benin,2782,Zogbodomey,561,"Fish (fresh, silvi) - Retail",0.0,XOF,15,Retail,5,KG,6,2021,1200.0,2021-06-01


In [7]:
# split des données
features = [
    'market_name',
    'commodity_name',
    'currency_name',
    'point_type_name',
    'unit_name',
    'month',
    'year'
]

X = benin_filtered[features]
y = benin_filtered['price']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [8]:
# encodage
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), features)], remainder='passthrough')

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)


In [ ]:
# recherche des meilleurs paramètres random forest regressor

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='neg_mean_absolute_error')

grid_search.fit(X_train_encoded, y_train)

print(f"Meilleurs hyperparamètres : {grid_search.best_params_}")

print(f"Meilleur score MAE : {-grid_search.best_score_}")



Fitting 5 folds for each of 24 candidates, totalling 120 fits


In [ ]:
# random forest model
random_forest_model = grid_search.best_estimator_

# entraînement avec les meilleurs hyperparamètres
random_forest_model.fit(X_train_encoded, y_train)

best_random_pred = random_forest_model.predict(X_test_encoded)

# évaluation
mae = mean_absolute_error(y_test, best_random_pred)
rmse = np.sqrt(mean_squared_error(y_test, best_random_pred))




In [ ]:
# recherche des meilleurs paramètres gradient boosting

param_grid_gradient_boosting_reg = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [5, 10]
}

gradient_boosting = GradientBoostingRegressor(random_state=42)

# appliquer le GridSearchCV
grid_search_gradient_boosting = GridSearchCV(
    estimator=gradient_boosting,
    param_grid=param_grid_gradient_boosting_reg,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='neg_mean_absolute_error'
   )

grid_search_gradient_boosting.fit(X_train_encoded, y_train)

print(f"Meilleurs hyperparamètres : {grid_search_gradient_boosting.best_params_}")

print(f"Meilleur score MAE : {-grid_search_gradient_boosting.best_score_}")


Fitting 5 folds for each of 32 candidates, totalling 160 fits
Meilleurs hyperparamètres : {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 5, 'n_estimators': 150}
Meilleur score MAE : 144.57271561823865


In [ ]:
# gradient boosting model

gradientBoosting_model = grid_search.best_estimator_

gradientBoosting_model.fit(X_train_encoded, y_train)

best_gb_pred = gradientBoosting_model.predict(X_test_encoded)

# évaluation
mae = mean_absolute_error(y_test, best_gb_pred)
rmse = np.sqrt(mean_squared_error(y_test, best_gb_pred))




In [ ]:
print(f'Par rapport au modèle RandomForestRegressor : \n')
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}\n")

print(f'Par rapport au modèle GradientBoostingRegressor : \n')
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")



Par rapport au modèle RandomForestRegressor : 

MAE : 188.37
RMSE : 509.25

Par rapport au modèle GradientBoostingRegressor : 

MAE : 188.37
RMSE : 509.25


In [ ]:
# importances = random_forest_model.feature_importances_
# feature_names = X_train_encoded.columns

# importance_df = pd.DataFrame({
#     'feature': feature_names,
#     'importance': importances
# }).sort_values(by='importance', ascending=False)

# # importance_df.head(10)

# # Prendre les 10 features les plus importantes
# top_features = importance_df.head(10)

# plt.figure(figsize=(10,6))
# sns.barplot(x='importance', y='feature', data=top_features, palette='viridis')
# plt.title("Top 10 des variables les plus importantes dans le Random Forest")
# plt.xlabel("Importance")
# plt.ylabel("Variable")
# plt.show()


### Conclusion


Les variables catégorielles ont été encodées, les données ont été séparées en jeux d’entraînement et de test (70/30), puis deux modèles non linéaires (Random Forest et Gradient Boosting) ont été optimisés par recherche d’hyperparamètres. Les performances sur le jeu de test étant équivalentes, le Random Forest a été retenu pour sa stabilité et sa robustesse.

- Le type de produit, le marché et la période (mois/année) sont les facteurs les plus déterminants des prix. Les produits comme l’huile sont systématiquement plus chers que les céréales, et certains marchés (Comé, Savalou) sont structurellement plus onéreux.

- Random Forest et Gradient Boosting donnent des performances identiques sur le jeu de test. Cela montre que la capacité du modèle à prédire le prix est limitée par l’information disponible : les chocs exogènes (crises, fermetures de frontières, transport, pandémie) ne sont pas inclus dans le jeu de données.

- Bien que les deux modèles non linéaires aient des performances équivalentes, le Random Forest est plus stable et robuste face aux variations et aux valeurs extrêmes. Il est donc retenu comme modèle final pour la prédiction statique des prix.
